# Retrieval Evaluation — Hands-On

Compute IR metrics from a tiny labeled retrieval set.

## 0. Setup

In [ ]:
%pip install -q numpy
import math, numpy as np
cases=[{"q":"refund policy","relevant":{"d1","d3"},"ranked":["d2","d1","d4","d3"]},{"q":"shipping delay","relevant":{"d4"},"ranked":["d4","d2","d1"]},{"q":"warranty","relevant":{"d5"},"ranked":["d2","d3","d1"]}]

## 1. Binary metrics

In [ ]:
def basic(ranked, relevant, k):
    top=ranked[:k]; inter=set(top)&relevant
    return {"hit":float(bool(inter)),"precision":len(inter)/k,"recall":len(inter)/len(relevant)}
for c in cases: print(c["q"], basic(c["ranked"], c["relevant"], 3))

## 2. MRR

In [ ]:
def rr(ranked, relevant):
    return next((1/(i+1) for i,d in enumerate(ranked) if d in relevant), 0.0)
print([rr(c["ranked"], c["relevant"]) for c in cases])
print("MRR", np.mean([rr(c["ranked"], c["relevant"]) for c in cases]))

## 3. NDCG with graded judgments

In [ ]:
def dcg(gains): return sum((2**g-1)/math.log2(i+2) for i,g in enumerate(gains))
def ndcg(ranked, grades, k):
    gains=[grades.get(d,0) for d in ranked[:k]]
    ideal=sorted(grades.values(), reverse=True)[:k]
    return dcg(gains)/(dcg(ideal) or 1)
grades={"d1":2,"d3":1}
print(ndcg(["d2","d1","d4","d3"], grades, 4))

## 4. Aggregate dashboard

In [ ]:
agg={"hit":[],"precision":[],"recall":[],"rr":[]}
for c in cases:
    b=basic(c["ranked"], c["relevant"], 3)
    for k in ["hit","precision","recall"]: agg[k].append(b[k])
    agg["rr"].append(rr(c["ranked"], c["relevant"]))
print({k:round(float(np.mean(v)),3) for k,v in agg.items()})

## 5. Exercise prompts
1. Compute metrics at k=1 and k=5.
2. Add query-type segments.
3. Compare two retrievers on the same cases.